# Book Notes Knowledge Worker

A private reading companion powered by your personal book notes.  
Ask questions about key ideas, frameworks, and concepts — grounded entirely in your own knowledge base.

**Books covered:** Atomic Habits · Deep Work · The Lean Startup  
**Stack:** OpenRouter (chat + chunking) · ChromaDB (vectors) · SentenceTransformers (embeddings) · Gradio (UI)

In [ ]:
%pip install -q chromadb sentence-transformers openai litellm pydantic gradio python-dotenv plotly scikit-learn tqdm

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from litellm import completion
from tqdm import tqdm
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import gradio as gr
from dotenv import load_dotenv

load_dotenv(override=True)

MODEL = 'openrouter/openai/gpt-4o-mini'
DB_NAME = 'book_notes_db'
COLLECTION_NAME = 'books'
EMBED_MODEL = 'all-MiniLM-L6-v2'
KB_PATH = Path('knowledge-base')
AVERAGE_CHUNK_SIZE = 500
RETRIEVAL_K = 8

In [2]:
sys.path.insert(0, '.')
from prompts.prompts import SYSTEM_PROMPT, CHUNK_PROMPT, REWRITE_PROMPT

In [3]:
def fetch_documents():
    documents = []
    for file in KB_PATH.glob('*.md'):
        book = file.stem.replace('_', ' ').title()
        documents.append({'book': book, 'source': file.as_posix(), 'text': file.read_text(encoding='utf-8')})
    print(f'Loaded {len(documents)} books: {[d["book"] for d in documents]}')
    return documents

documents = fetch_documents()

Loaded 3 books: ['Atomic Habits', 'The Lean Startup', 'Deep Work']


In [4]:
class Result(BaseModel):
    page_content: str
    metadata: dict


class Chunk(BaseModel):
    headline: str = Field(description='A brief heading for this chunk')
    summary: str = Field(description='2-3 sentences summarising the chunk content')
    original_text: str = Field(description='The original text of this chunk, exactly as written')

    def as_result(self, document):
        content = self.headline + '\n\n' + self.summary + '\n\n' + self.original_text
        return Result(page_content=content, metadata={'source': document['source'], 'book': document['book']})


class Chunks(BaseModel):
    chunks: list[Chunk]

In [5]:
def make_prompt(document):
    how_many = (len(document['text']) // AVERAGE_CHUNK_SIZE) + 1
    return CHUNK_PROMPT.format(book=document['book'], how_many=how_many, text=document['text'])


def process_document(document):
    messages = [{'role': 'user', 'content': make_prompt(document)}]
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    return [chunk.as_result(document) for chunk in Chunks.model_validate_json(response.choices[0].message.content).chunks]


def create_chunks(documents):
    all_chunks = []
    for doc in tqdm(documents):
        all_chunks.extend(process_document(doc))
    return all_chunks


chunks = create_chunks(documents)
print(f'Total chunks: {len(chunks)}')

100%|██████████| 3/3 [01:36<00:00, 32.31s/it]

Total chunks: 36


In [6]:
chunks

[Result(page_content='Overview of Atomic Habits\n\nAtomic Habits is a guide on building good habits and breaking bad ones, emphasizing the significance of tiny 1% improvements. The foundational idea is that habits, much like atoms, are small yet essential to our lives. It argues that success stems from effective systems of daily habits rather than merely aiming for goals.\n\nAtomic Habits is a practical guide to building good habits and breaking bad ones. James Clear argues that tiny 1% improvements compound over time into remarkable results. The title comes from the idea that habits are the *atoms* of our lives — small and often invisible, but foundational to everything.\n\nThe core thesis: you do not rise to the level of your goals, you fall to the level of your systems. Winners and losers have the same goals. What differs is the system of daily habits.', metadata={'source': 'knowledge-base/atomic_habits.md', 'book': 'Atomic Habits'}),
 Result(page_content="The 1% Better Principle\n\

In [7]:
def create_embeddings(chunks):
    embed_fn = SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
    chroma = PersistentClient(path=DB_NAME)
    if COLLECTION_NAME in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(COLLECTION_NAME)
    collection = chroma.get_or_create_collection(COLLECTION_NAME, embedding_function=embed_fn)
    collection.add(
        ids=[str(i) for i in range(len(chunks))],
        documents=[c.page_content for c in chunks],
        metadatas=[c.metadata for c in chunks],
    )
    print(f'Vectorstore ready — {collection.count()} chunks stored')
    return collection


collection = create_embeddings(chunks)

README.md: 0.00B [00:00, ?B/s]

Vectorstore ready — 36 chunks stored


In [8]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
books = [m['book'] for m in result['metadatas']]
unique_books = list(set(books))

palette = ['#4C8BF5', '#34A853', '#EA4335']
color_map = {book: palette[i] for i, book in enumerate(unique_books)}
colors = [color_map[b] for b in books]

tsne = TSNE(n_components=2, random_state=42)
reduced = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced[:, 0],
    y=reduced[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=['Book: ' + b + '<br>' + d[:80] + '...' for b, d in zip(books, result['documents'])],
    hoverinfo='text',
)])
fig.update_layout(title='Book Notes — Vector Space (t-SNE 2D)', width=800, height=500)
fig.show()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Advanced RAG

Three techniques on top of basic vector search:
1. **Query rewriting** — restate the user's question as a precise KB search query
2. **Reranking** — LLM re-orders retrieved chunks by relevance before answering
3. **Conversation history** — query rewriter has access to prior turns for context

In [9]:
class RankOrder(BaseModel):
    order: list[int] = Field(description='Chunk IDs ranked from most to least relevant to the question')


def rerank(question, chunks):
    user_prompt = 'Question: ' + question + '\n\n'
    user_prompt += '\n\n'.join('# CHUNK ' + str(i + 1) + ':\n' + c.page_content for i, c in enumerate(chunks))
    messages = [
        {'role': 'system', 'content': 'Re-rank document chunks by relevance to the question. Reply only with the ranked list of chunk IDs.'},
        {'role': 'user', 'content': user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    order = RankOrder.model_validate_json(response.choices[0].message.content).order
    return [chunks[i - 1] for i in order if 0 < i <= len(chunks)]


def fetch_context_unranked(question):
    embed_fn = SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
    chroma = PersistentClient(path=DB_NAME)
    coll = chroma.get_or_create_collection(COLLECTION_NAME, embedding_function=embed_fn)
    results = coll.query(query_texts=[question], n_results=RETRIEVAL_K)
    return [Result(page_content=doc, metadata=meta)
            for doc, meta in zip(results['documents'][0], results['metadatas'][0])]


def fetch_context(question):
    return rerank(question, fetch_context_unranked(question))

In [10]:
def rewrite_query(question, history):
    history_text = '\n'.join(
        ('User' if m['role'] == 'user' else 'Assistant') + ': ' + m['content']
        for m in history[-4:]
    )
    message = REWRITE_PROMPT.format(history=history_text, question=question)
    response = completion(model=MODEL, messages=[{'role': 'system', 'content': message}])
    return response.choices[0].message.content.strip()

In [11]:
def make_rag_messages(question, history, chunks):
    context = '\n\n'.join('Extract from ' + c.metadata['book'] + ':\n' + c.page_content for c in chunks)
    system = SYSTEM_PROMPT.format(context=context)
    return [{'role': 'system', 'content': system}] + history + [{'role': 'user', 'content': question}]


def chat(message, history):
    query = rewrite_query(message, history) if history else message
    print(f'Query: {query}')
    chunks = fetch_context(query)
    messages = make_rag_messages(message, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [12]:
gr.ChatInterface(
    fn=chat,
    type='messages',
    title='📚 Book Notes Knowledge Worker',
    description='Ask me anything about Atomic Habits, Deep Work, or The Lean Startup.',
    examples=[
        'What is habit stacking and how does it work?',
        'How does Cal Newport define deep work?',
        'What is the Build-Measure-Learn loop?',
        'What are the Four Laws of Behaviour Change?',
        'What is the Two-Minute Rule?',
    ],
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Query: What is habit stacking and how does it work?
